# Pneumonia Detection from Chest X-rays: DenseNet121 + Grad-CAM

Everything in one notebook (model, dataset, evaluation, Grad-CAM, box localization, inference).
The best checkpoint is loaded from Google Drive, so **no training is needed**.

**How to use:** *Runtime → Run all*, then upload an X-ray in the last cell.

| Section | Purpose |
|---|---|
| 1 | Setup (Drive, installs, paths) |
| 2 | Model definition |
| 3 | Dataset (only needed for evaluation) |
| 4 | Grad-CAM utilities |
| 5 | Load the trained checkpoint |
| 6 | *(Optional)* Evaluate on `test.csv` |
| 7 | Prediction + Grad-CAM function |
| 8 | **Localization: Grad-CAM boxes vs radiologist boxes (IoU)** |
| 9 | **Upload an image and predict** |

## 1. Setup

In [ ]:
!pip install -q pydicom

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ----------------------------- CONFIG -----------------------------
PROJECT_DIR     = "/content/drive/MyDrive/Pneumonia_Project"
CHECKPOINT_PATH = f"{PROJECT_DIR}/checkpoints/best_model.pth"
TEST_CSV        = f"{PROJECT_DIR}/metadata/test.csv"
RESULTS_DIR     = f"{PROJECT_DIR}/results"
IMG_DIR         = "/content/rsna_train_images/stage_2_train_images/"   # only needed for section 6

THRESHOLD       = 0.5     # probability >= THRESHOLD  ->  Pneumonia
RUN_EVALUATION  = False   # set True to run section 6 (needs test.csv and the DICOM folder)
# ------------------------------------------------------------------

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Checkpoint exists:", os.path.isfile(CHECKPOINT_PATH))

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from scipy import ndimage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Model

In [ ]:
class PneumoniaDenseNet121(nn.Module):
    """DenseNet121 with a single-logit head (pair with BCEWithLogitsLoss / sigmoid)."""
    def __init__(self, pretrained: bool = True, drop_rate: float = 0.2):
        super().__init__()
        try:
            from torchvision.models import DenseNet121_Weights
            weights = DenseNet121_Weights.DEFAULT if pretrained else None
            self.densenet = models.densenet121(weights=weights)
        except (ImportError, AttributeError):
            self.densenet = models.densenet121(pretrained=pretrained)

        in_features = self.densenet.classifier.in_features   # 1024
        if drop_rate > 0:
            self.densenet.classifier = nn.Sequential(nn.Dropout(p=drop_rate),
                                                     nn.Linear(in_features, 1))
        else:
            self.densenet.classifier = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.densenet(x)          # (batch, 1) raw logits


def get_model(pretrained: bool = True, drop_rate: float = 0.2):
    return PneumoniaDenseNet121(pretrained=pretrained, drop_rate=drop_rate)

## 3. Dataset

In [ ]:
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


def read_xray(image_path):
    """Read a DICOM X-ray and convert it to an 8-bit grayscale array."""
    dicom = pydicom.dcmread(str(image_path))
    image = dicom.pixel_array.astype(np.float32)

    slope = float(getattr(dicom, "RescaleSlope", 1))
    intercept = float(getattr(dicom, "RescaleIntercept", 0))
    image = image * slope + intercept

    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() - image          # low values displayed as white

    image -= image.min()
    if image.max() > 0:
        image = image / image.max() * 255.0
    return image.astype(np.uint8)


def get_eval_transform():
    """Deterministic preprocessing, identical to validation/test during training."""
    return T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD),
    ])


def load_image(image_path):
    """Load a DICOM, PNG or JPG X-ray as an RGB PIL image."""
    image_path = Path(image_path)
    if image_path.suffix.lower() == ".dcm":
        return Image.fromarray(read_xray(image_path)).convert("RGB")
    with Image.open(image_path) as source:
        return source.convert("RGB")


class PneumoniaDataset(Dataset):
    """Binary pneumonia classification dataset (RSNA)."""
    def __init__(self, metadata, image_dir, transform=None,
                 patient_col="patientId", label_col="Target"):
        if isinstance(metadata, (str, Path)):
            metadata = pd.read_csv(metadata)
        self.metadata = metadata.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.transform = transform or get_eval_transform()
        self.patient_col = patient_col
        self.label_col = label_col

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        patient_id = str(row[self.patient_col])

        image_path = self.image_dir / f"{patient_id}.dcm"
        if not image_path.exists():
            image_path = self.image_dir / patient_id
        if not image_path.exists():
            raise FileNotFoundError(f"X-ray not found: {self.image_dir / f'{patient_id}.dcm'}")

        image = load_image(image_path)
        image = self.transform(image)
        label = torch.tensor(float(row[self.label_col]), dtype=torch.float32)
        return image, label, patient_id

## 4. Grad-CAM

In [ ]:
TARGET_LAYERS = ("norm5", "denseblock3")


class GradCAM:
    """Grad-CAM for a single-logit DenseNet pneumonia classifier."""

    def __init__(self, model, target_layers=("norm5",)):
        if not target_layers:
            raise ValueError("At least one target layer must be provided.")
        unknown = set(target_layers) - set(TARGET_LAYERS)
        if unknown:
            raise ValueError(f"Unsupported target layer(s): {unknown}; choose from {TARGET_LAYERS}")
        if not hasattr(model, "densenet"):
            raise AttributeError("Expected model.densenet")
        self.model = model
        self.densenet = model.densenet
        self.target_layers = tuple(target_layers)

    def compute(self, input_tensor):
        """input_tensor: (1, 3, H, W). Returns ({layer: HxW CAM in [0,1]}, P(pneumonia))."""
        if input_tensor.ndim != 4 or input_tensor.shape[0] != 1:
            raise ValueError("Expected input shape (1, 3, H, W).")

        self.model.eval()
        captured = {}

        # Replay DenseNet's forward pass so intermediate maps keep their gradients.
        with torch.enable_grad():
            h = input_tensor
            for name, module in self.densenet.features.named_children():
                h = module(h)
                if name == "norm5":
                    h = F.relu(h)
                if name in self.target_layers:
                    captured[name] = h

            missing = set(self.target_layers) - set(captured)
            if missing:
                raise RuntimeError(f"Could not capture target layers: {missing}")

            pooled = F.adaptive_avg_pool2d(h, (1, 1)).flatten(1)
            logits = self.densenet.classifier(pooled)
            grads = torch.autograd.grad(
                logits.sum(), [captured[n] for n in self.target_layers]
            )

        cams = {}
        for name, grad in zip(self.target_layers, grads):
            activation = captured[name]
            weights = grad.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * activation).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=input_tensor.shape[-2:],
                                mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
            cam = np.asarray(cam, dtype=np.float32)
            cam -= float(cam.min())
            if float(cam.max()) > 0:
                cam /= float(cam.max())
            cams[name] = cam

        return cams, torch.sigmoid(logits).item()


def resize_cam(cam, size):
    """Resize a 2D CAM to (width, height) keeping values in [0, 1]."""
    cam = np.clip(np.asarray(cam, dtype=np.float32), 0.0, 1.0)
    img = Image.fromarray(np.round(cam * 255).astype(np.uint8))
    img = img.resize(size, Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def explain(model, pil_image, device=None, target_layer="norm5", flip_tta=False):
    """Return (CAM at original image size, pneumonia probability)."""
    if target_layer not in TARGET_LAYERS:
        raise ValueError(f"target_layer must be one of {TARGET_LAYERS}")
    device = device or next(model.parameters()).device
    tensor = get_eval_transform()(pil_image).unsqueeze(0).to(device)

    gradcam = GradCAM(model, (target_layer,))
    cams, probability = gradcam.compute(tensor)
    cam = cams[target_layer]

    if flip_tta:   # average with the mirrored image's CAM (flipped back)
        flipped, _ = gradcam.compute(torch.flip(tensor, dims=[3]))
        cam = (cam + flipped[target_layer][:, ::-1]) / 2.0
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = cam.astype(np.float32)

    return resize_cam(cam, pil_image.size), probability


def overlay_heatmap(pil_image, cam, alpha=0.4, colormap="jet"):
    """Blend a CAM over the X-ray. An all-zero CAM returns the image unchanged."""
    base = np.asarray(pil_image.convert("RGB"), dtype=np.float32)
    if float(cam.max()) <= 0:
        return base.astype(np.uint8)
    cam = np.clip(cam, 0.0, 1.0)
    heat = (plt.get_cmap(colormap)(cam)[..., :3] * 255).astype(np.float32)
    return np.clip((1 - alpha) * base + alpha * heat, 0, 255).astype(np.uint8)


def cam_to_boxes(cam, threshold=0.5, min_area_frac=0.01, max_boxes=3, split_midline=False):
    """
    Approximate pneumonia boxes from a [0,1] heatmap (from localization.py):
    threshold -> 8-connected regions -> drop tiny ones -> keep top max_boxes by activation.
    Returns dicts {x, y, width, height, score} (x, y = top-left).
    """
    mask = cam >= threshold
    if split_midline:
        mid = cam.shape[1] // 2
        mask[:, mid - 1:mid + 1] = False
    labels, n = ndimage.label(mask, structure=np.ones((3, 3), dtype=int))
    if n == 0:
        return []

    min_area = min_area_frac * cam.size
    index = np.arange(1, n + 1)
    areas = ndimage.sum(mask, labels, index)
    masses = ndimage.sum(cam, labels, index)
    boxes = []
    for area, mass, (ys, xs) in zip(areas, masses, ndimage.find_objects(labels)):
        if area < min_area:
            continue
        boxes.append({"x": float(xs.start), "y": float(ys.start),
                      "width": float(xs.stop - xs.start),
                      "height": float(ys.stop - ys.start),
                      "score": float(mass)})
    boxes.sort(key=lambda b: b["score"], reverse=True)
    return boxes[:max_boxes]

## 5. Load the trained model from Drive

In [ ]:
def load_model(checkpoint_path, device):
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")
    # weights_only=False: your checkpoint also stores optimizer state / numpy scalars.
    # Only load checkpoints you trust.
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model = get_model(pretrained=False).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    if isinstance(ckpt, dict):
        print(f"Loaded checkpoint (epoch {ckpt.get('epoch', '?')}, val AUC {ckpt.get('val_auc', float('nan')):.4f})")
    return model

model = load_model(CHECKPOINT_PATH, device)

## 6. *(Optional)* Evaluate on the held-out test set
Set `RUN_EVALUATION = True` in the config cell. This needs `test.csv` on Drive and the RSNA DICOMs in `IMG_DIR`.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

def run_evaluation(model, test_csv, img_dir, results_dir, threshold=0.5, batch_size=32):
    ds = PneumoniaDataset(test_csv, img_dir, transform=get_eval_transform())
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    print(f"Test set: {len(ds)} cases")

    targets, probs = [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            p = torch.sigmoid(model(images.to(device)))
            targets.extend(labels.numpy().flatten())
            probs.extend(p.cpu().numpy().flatten())

    y = np.array(targets).astype(int)
    p = np.array(probs)
    pred = (p >= threshold).astype(int)

    metrics = {
        "accuracy":  float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall":    float(recall_score(y, pred, zero_division=0)),
        "f1_score":  float(f1_score(y, pred, zero_division=0)),
        "roc_auc":   float(roc_auc_score(y, p)),
        "confusion_matrix": confusion_matrix(y, pred).tolist(),
    }
    with open(os.path.join(results_dir, "test_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)

    cm = np.array(metrics["confusion_matrix"])
    fpr, tpr, _ = roc_curve(y, p)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    im = ax[0].imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax[0])
    labels_ = ["Normal (0)", "Pneumonia (1)"]
    ax[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=labels_, yticklabels=labels_,
              title="Confusion Matrix (test)", xlabel="Predicted", ylabel="True")
    for i in range(2):
        for j in range(2):
            ax[0].text(j, i, int(cm[i, j]), ha="center", va="center",
                       color="white" if cm[i, j] > cm.max() / 2 else "black")

    ax[1].plot(fpr, tpr, color="crimson", lw=2.5, label=f"DenseNet121 (AUC = {metrics['roc_auc']:.4f})")
    ax[1].plot([0, 1], [0, 1], "--", color="navy", label="Chance")
    ax[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
    ax[1].legend(loc="lower right"); ax[1].grid(True, ls=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "evaluation_plots.png"), dpi=200)
    plt.show()

    print("=" * 44)
    for k in ("accuracy", "precision", "recall", "f1_score", "roc_auc"):
        print(f"  {k:<10}: {metrics[k]:.4f}")
    print("  Confusion matrix [[TN, FP], [FN, TP]]:\n", cm)
    print("=" * 44)
    print("Saved to:", results_dir)
    return metrics

if RUN_EVALUATION:
    metrics = run_evaluation(model, TEST_CSV, IMG_DIR, RESULTS_DIR, THRESHOLD)
else:
    print("Skipping evaluation (RUN_EVALUATION = False).")

## 7. Prediction + Grad-CAM function

In [ ]:
def predict_and_explain(image_path, threshold=THRESHOLD, target_layer="norm5",
                        flip_tta=False, show_boxes=True, box_threshold=0.5, save_to=None):
    """Classify one X-ray, print the scores, and show the Grad-CAM heatmap."""
    image = load_image(image_path)
    cam, prob = explain(model, image, device, target_layer, flip_tta)

    label = "PNEUMONIA" if prob >= threshold else "NORMAL"
    confidence = prob if prob >= threshold else 1 - prob

    print("=" * 44)
    print(f"  Prediction        : {label}")
    print(f"  P(pneumonia)      : {prob:.4f}  ({prob*100:.1f}%)")
    print(f"  P(normal)         : {1-prob:.4f}  ({(1-prob)*100:.1f}%)")
    print(f"  Confidence        : {confidence*100:.1f}%")
    print(f"  Decision threshold: {threshold}")
    print(f"  Grad-CAM layer    : {target_layer}{' + flip TTA' if flip_tta else ''}")
    print("=" * 44)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), gridspec_kw={"width_ratios": [1, 1, 0.7]})

    axes[0].imshow(image); axes[0].set_title("Input X-ray"); axes[0].axis("off")

    axes[1].imshow(overlay_heatmap(image, cam))
    if float(cam.max()) <= 0:
        axes[1].set_title("No positive Grad-CAM evidence")
    else:
        axes[1].set_title(f"Grad-CAM ({target_layer})")
        if show_boxes and prob >= threshold:
            for b in cam_to_boxes(cam, threshold=box_threshold):
                axes[1].add_patch(plt.Rectangle((b["x"], b["y"]), b["width"], b["height"],
                                                fill=False, edgecolor="lime", linewidth=2.5))
    axes[1].axis("off")

    colors = ["#2e8b57", "#c0392b"]
    bars = axes[2].barh(["Normal", "Pneumonia"], [1 - prob, prob], color=colors)
    axes[2].axvline(threshold, color="gray", ls=":", lw=1)
    axes[2].set_xlim(0, 1); axes[2].set_xlabel("Probability")
    axes[2].set_title(f"Prediction: {label}", color=colors[1] if label == "PNEUMONIA" else colors[0],
                      fontweight="bold")
    for bar, v in zip(bars, [1 - prob, prob]):
        axes[2].text(min(v + 0.02, 0.8), bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center")

    plt.tight_layout()
    if save_to:
        fig.savefig(save_to, dpi=150, bbox_inches="tight")
    plt.show()
    return {"label": label, "prob_pneumonia": prob, "cam": cam}

## 8. Localization: Grad-CAM boxes vs radiologist boxes (IoU)
Uses only the saved model (no training, no classifier re-evaluation), so it runs fine on **CPU**.
1. Computes Grad-CAM for pneumonia-positive patients from `val.csv`.
2. Picks the best heatmap threshold **on val**.
3. Reports IoU on `test.csv` **once** with that threshold.

Needs: `all_annotations.csv` (RSNA box labels) in `metadata/`, and the DICOMs in `IMG_DIR`.
Lower `N_IMAGES` if it is too slow.

In [ ]:
ANNOTATIONS_CSV = f"{PROJECT_DIR}/metadata/all_annotations.csv"
VAL_CSV         = f"{PROJECT_DIR}/metadata/val.csv"
N_IMAGES        = 100              # positives per split (CPU: ~100 is a few minutes)
LOC_LAYER       = "norm5"          # or "denseblock3"
THRESHOLDS      = [0.3, 0.4, 0.5, 0.6, 0.7]
N_EXAMPLES      = 6

def load_gt_boxes(annotations_csv):
    ann = pd.read_csv(annotations_csv)
    ann = ann[ann["Target"] == 1].dropna(subset=["x", "y", "width", "height"])
    gt = {}
    for r in ann.itertuples(index=False):
        gt.setdefault(r.patientId, []).append(
            {"x": float(r.x), "y": float(r.y), "width": float(r.width), "height": float(r.height)})
    return gt

def boxes_to_mask(boxes, shape):
    m = np.zeros(shape, dtype=bool)
    for b in boxes:
        x1, y1 = int(round(b["x"])), int(round(b["y"]))
        x2, y2 = int(round(b["x"] + b["width"])), int(round(b["y"] + b["height"]))
        m[max(y1, 0):max(y2, 0), max(x1, 0):max(x2, 0)] = True
    return m

def union_iou(pred, gt, shape):
    p, g = boxes_to_mask(pred, shape), boxes_to_mask(gt, shape)
    u = np.logical_or(p, g).sum()
    return float(np.logical_and(p, g).sum() / u) if u > 0 else 0.0

def peak_in_gt(cam, gt):
    y, x = np.unravel_index(np.argmax(cam), cam.shape)
    return any(b["x"] <= x < b["x"] + b["width"] and b["y"] <= y < b["y"] + b["height"] for b in gt)

def collect(split_csv, gt, n):
    # Heatmaps are computed once per image, then reused for every threshold.
    df = pd.read_csv(split_csv)
    pids = [p for p in df.loc[df["Target"] == 1, "patientId"] if p in gt][:n]
    items = []
    for i, pid in enumerate(pids, 1):
        img = load_image(os.path.join(IMG_DIR, f"{pid}.dcm"))
        cam, prob = explain(model, img, device, LOC_LAYER)
        items.append({"pid": pid, "cam": cam, "prob": prob, "gt": gt[pid],
                      "image": img if i <= N_EXAMPLES else None})
        if i % 25 == 0 or i == len(pids):
            print(f"  {i}/{len(pids)} images processed")
    return items

def score(items, thr):
    rows = []
    for it in items:
        boxes = cam_to_boxes(it["cam"], threshold=thr)
        rows.append({"patientId": it["pid"], "prob": it["prob"], "n_pred_boxes": len(boxes),
                     "mask_iou": union_iou(boxes, it["gt"], it["cam"].shape),
                     "detected": it["prob"] >= THRESHOLD,
                     "pointing_hit": int(peak_in_gt(it["cam"], it["gt"]))})
    d = pd.DataFrame(rows)
    d["iou_paper_style"] = np.where(d["detected"], d["mask_iou"], 0.0)   # missed pneumonia = IoU 0
    return d

gt_boxes = load_gt_boxes(ANNOTATIONS_CSV)

print("VAL: computing heatmaps ...")
val_items = collect(VAL_CSV, gt_boxes, N_IMAGES)
sweep = pd.DataFrame([{"threshold": t, "mean_mask_iou": score(val_items, t)["mask_iou"].mean()}
                      for t in THRESHOLDS])
best_thr = float(sweep.loc[sweep["mean_mask_iou"].idxmax(), "threshold"])
print("\nThreshold sweep on VAL:\n", sweep.to_string(index=False))
print(f"\n-> Best threshold chosen on val: {best_thr}")

In [ ]:
print("TEST: computing heatmaps ...")
test_items = collect(TEST_CSV, gt_boxes, N_IMAGES)
res = score(test_items, best_thr)

loc_dir = os.path.join(RESULTS_DIR, "localization")
os.makedirs(loc_dir, exist_ok=True)
res.to_csv(os.path.join(loc_dir, "localization_test_per_patient.csv"), index=False)

summary = {
    "layer": LOC_LAYER, "threshold": best_thr, "n_patients": int(len(res)),
    "mean_mask_iou_all_true_positives": float(res["mask_iou"].mean()),
    "mean_iou_paper_style": float(res["iou_paper_style"].mean()),
    "pointing_game": float(res["pointing_hit"].mean()),
    "classifier_sensitivity_on_these": float(res["detected"].mean()),
    "images_with_no_box": int((res["n_pred_boxes"] == 0).sum()),
}
with open(os.path.join(loc_dir, "localization_test_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("=" * 54)
print(f"  LOCALIZATION ON TEST  ({len(res)} pneumonia patients)")
print("=" * 54)
print(f"  Layer / threshold            : {LOC_LAYER} / {best_thr}")
print(f"  Mean IoU (all true positives): {summary['mean_mask_iou_all_true_positives']:.4f}")
print(f"  Mean IoU (paper style)       : {summary['mean_iou_paper_style']:.4f}   <- missed cases count as 0")
print(f"  Pointing game (peak in box)  : {summary['pointing_game']:.3f}")
print(f"  Classifier sensitivity       : {summary['classifier_sensitivity_on_these']:.3f}")
print("  Paper reference (CAM+CNN)    : 0.1508 IoU")
print("=" * 54)

# Example figure: green = radiologist box, red = Grad-CAM box
ex = [it for it in test_items if it["image"] is not None][:N_EXAMPLES]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, it in zip(axes.flat, ex):
    boxes = cam_to_boxes(it["cam"], threshold=best_thr)
    ax.imshow(overlay_heatmap(it["image"], it["cam"]))
    for b in it["gt"]:
        ax.add_patch(plt.Rectangle((b["x"], b["y"]), b["width"], b["height"], fill=False, edgecolor="lime", lw=3))
    for b in boxes:
        ax.add_patch(plt.Rectangle((b["x"], b["y"]), b["width"], b["height"], fill=False, edgecolor="red", lw=3))
    ax.set_title(f"{it['pid'][:8]}  P={it['prob']:.2f}  IoU={union_iou(boxes, it['gt'], it['cam'].shape):.2f}")
    ax.axis("off")
fig.suptitle("Green = radiologist ground truth | Red = Grad-CAM box", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(loc_dir, "localization_test_examples.png"), dpi=120)
plt.show()
print("Saved to:", loc_dir)

## 9. Upload an X-ray and predict
Supports **.dcm**, **.png**, **.jpg**. Run this cell again for each new image.

> ⚠️ Research/educational tool only, not a medical device. Grad-CAM and the green boxes are approximate visual explanations, not diagnoses.

In [ ]:
from google.colab import files

uploaded = files.upload()

for fname in uploaded:
    print(f"\n>>> {fname}")
    predict_and_explain(
        fname,
        target_layer="norm5",     # or "denseblock3" (finer, 14x14)
        flip_tta=False,           # True = average CAM with the mirrored image
        show_boxes=True,          # green boxes from the heatmap (only drawn if pneumonia predicted)
    )